In [1]:
import os
from pathlib import Path

os.chdir(Path().resolve().parent)

In [36]:
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import re
import numpy as np
import torch
from torch_geometric.data import InMemoryDataset, Data
import pandas as pd


HDR_RE = re.compile(r"CSD_code\s*=\s*([A-Za-z0-9_+-]+)")

def _extract_code(header_line: str) -> Optional[str]:
    m = HDR_RE.search(header_line)
    return m.group(1) if m else None

# --- periodic table (extend if needed) ---
PERIODIC = {
    'H': 1,  'He': 2,
    'Li': 3, 'Be': 4, 'B': 5,  'C': 6,  'N': 7,  'O': 8,  'F': 9,  'Ne': 10,
    'Na': 11,'Mg': 12,'Al': 13,'Si': 14,'P': 15,'S': 16,'Cl': 17,'Ar': 18,
    'K': 19, 'Ca': 20,
    # transition metals (1st, 2nd, 3rd rows)
    'Sc': 21,'Ti': 22,'V': 23,'Cr': 24,'Mn': 25,'Fe': 26,'Co': 27,'Ni': 28,'Cu': 29,'Zn': 30,
    'Ga': 31,'Ge': 32,'As': 33,'Se': 34,'Br': 35,'Kr': 36,
    'Rb': 37,'Sr': 38,
    'Y': 39,'Zr': 40,'Nb': 41,'Mo': 42,'Tc': 43,'Ru': 44,'Rh': 45,'Pd': 46,'Ag': 47,'Cd': 48,
    'In': 49,'Sn': 50,'Sb': 51,'Te': 52,'I': 53,'Xe': 54,
    'Cs': 55,'Ba': 56,
    # lanthanides
    'La': 57,'Ce': 58,'Pr': 59,'Nd': 60,'Pm': 61,'Sm': 62,'Eu': 63,'Gd': 64,
    'Tb': 65,'Dy': 66,'Ho': 67,'Er': 68,'Tm': 69,'Yb': 70,'Lu': 71,
    # 5th period transition metals
    'Hf': 72,'Ta': 73,'W': 74,'Re': 75,'Os': 76,'Ir': 77,'Pt': 78,'Au': 79,'Hg': 80,
    'Tl': 81,'Pb': 82,'Bi': 83,'Po': 84,'At': 85,'Rn': 86,
    # actinides (common ones)
    'U': 92,'Pu': 94
}

def sym2Z(sym: str) -> int:
    z = PERIODIC.get(sym)
    if z is None:
        raise ValueError(f"Unknown element symbol: {sym}")
    return z

# --- file readers ---

# def read_xyz(path: Path) -> List[Tuple[List[str], np.ndarray]]:
#     """
#     Reads one or more molecules from an XYZ file.
#     Returns a list of (symbols, positions[N,3]) tuples.
#     """
#     lines = [l.strip() for l in path.read_text().splitlines() if l.strip()]
#     i = 0
#     molecules = []
#     while i < len(lines):
#         try:
#             n = int(lines[i].split()[0])
#         except ValueError:
#             break  # malformed block
#         start = i + 2  # skip count and comment
#         block = lines[start:start + n]
#         if len(block) < n:
#             break
#         symbols, coords = [], []
#         for line in block:
#             parts = line.split()
#             if len(parts) < 4:
#                 continue
#             symbols.append(parts[0])
#             coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
#         molecules.append((symbols, np.array(coords, dtype=np.float32)))
#         i = start + n
#     return molecules

# def read_q_file(path: Path) -> np.ndarray:
#     """Reads per-atom scalar charges: one value per line or whitespace-separated."""
#     vals = [float(v) for v in path.read_text().split()]
#     return np.asarray(vals, dtype=np.float32)

# def read_bo(path: Path, n_atoms: int, offset: int = 0) -> Tuple[np.ndarray, np.ndarray]:
#     """
#     Reads a simple bond order edge list: 'i j bo' per line (1-based typical).
#     Returns directed edges with both directions. Applies 'offset' to atom indices when
#     concatenating multiple parts.
#     """
#     src, dst, w = [], [], []
#     for line in path.read_text().splitlines():
#         parts = line.split()
#         if len(parts) < 3: 
#             continue
#         i, j = int(parts[0]) - 1 + offset, int(parts[1]) - 1 + offset
#         if not (offset <= i < offset + n_atoms and offset <= j < offset + n_atoms):
#             # tolerate files that include cross-part indices; keep them if within global range later
#             pass
#         bo = float(parts[2])
#         src += [i, j]
#         dst += [j, i]
#         w   += [bo, bo]
#     if not src:
#         return (np.zeros((2, 0), dtype=np.int64), np.zeros((0, 1), dtype=np.float32))
#     edge_index = np.vstack([np.asarray(src, np.int64), np.asarray(dst, np.int64)])
#     edge_attr  = np.asarray(w, dtype=np.float32)[:, None]
#     return edge_index, edge_attr

# # --- grouping helpers ---


def read_xyz_multi_by_code(path: Path) -> Dict[str, Tuple[List[str], np.ndarray]]:
    """
    Returns {code: (symbols, pos[N,3])} for all blocks in this file.
    Format per block:
      <N>
      CSD_code = <CODE> | ...
      <N atom lines>
    """
    lines = [l.rstrip() for l in path.read_text().splitlines()]
    i, out = 0, {}
    L = len(lines)
    while i < L:
        # skip empties
        while i < L and not lines[i].strip():
            i += 1
        if i >= L:
            break
        # count line
        try:
            n = int(lines[i].split()[0])
        except Exception:
            # not a block start -> skip and continue
            i += 1
            continue
        if i + 1 >= L: break
        header = lines[i+1]
        code = _extract_code(header) or f"{path.stem}_offset{i}"
        start = i + 2
        block = lines[start:start+n]
        if len(block) < n:  # incomplete
            break
        symbols, coords = [], []
        for ln in block:
            parts = ln.split()
            if len(parts) < 4: continue
            symbols.append(parts[0])
            coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
        if symbols:
            out[code] = (symbols, np.asarray(coords, dtype=np.float32))
        i = start + n
    return out

# --- parse multi-molecule BO by CSD_code ---
def read_bo_multi_by_code(path: Path) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """
    Returns {code: (edge_index[2,E], edge_attr[E,1])}.
    BO block starts with a header line "CSD_code = CODE ..." then atom lines:
      idx sym <float>  (sym idx bo)*
    We collect (idx, bo) pairs; indices are 1-based in file -> convert to 0-based.
    """
    lines = [l.rstrip() for l in path.read_text().splitlines()]
    out: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    code = None
    src, dst, w = [], [], []
    def _flush():
        nonlocal src, dst, w, code
        if code is not None:
            if src:
                ei = np.vstack([np.asarray(src, np.int64), np.asarray(dst, np.int64)])
                ea = np.asarray(w, dtype=np.float32)[:, None]
            else:
                ei = np.zeros((2,0), dtype=np.int64)
                ea = np.zeros((0,1), dtype=np.float32)
            out[code] = (ei, ea)
        src, dst, w = [], [], []

    for ln in lines:
        if not ln.strip():
            continue
        if "CSD_code" in ln:
            _flush()
            code = _extract_code(ln)
            continue
        # atom connectivity line
        toks = ln.split()
        if not toks or not toks[0].isdigit():
            continue
        # format: i  Sym  <float>  [Sym j bo]...
        try:
            i_idx = int(toks[0]) - 1
        except Exception:
            continue
        # neighbors start at column 3 (0-based idx 3) because col 1 = sym, col 2 = float
        k = 3
        while k + 2 < len(toks):
            # toks[k]   = neighbor symbol (ignored)
            # toks[k+1] = neighbor index
            # toks[k+2] = bo
            try:
                j_idx = int(toks[k+1]) - 1
                bo = float(toks[k+2])
            except Exception:
                break
            src.extend([i_idx, j_idx])
            dst.extend([j_idx, i_idx])
            w.extend([bo, bo])
            k += 3
    _flush()
    return out

# --- parse multi-molecule charges by CSD_code ---
def read_q_multi_by_code(path: Path) -> Dict[str, np.ndarray]:
    """
    Returns {code: charges[N]}.
    Block:
      CSD_code = CODE ...
      elem val
      ...
      Total charge = X
    We ignore the element symbol and keep values in order.
    """
    lines = [l.rstrip() for l in path.read_text().splitlines()]
    out: Dict[str, np.ndarray] = {}
    code = None
    vals: List[float] = []
    def _flush():
        nonlocal vals, code
        if code is not None and vals:
            out[code] = np.asarray(vals, dtype=np.float32)
        vals = []
    for ln in lines:
        if not ln.strip():
            continue
        if "CSD_code" in ln:
            _flush()
            code = _extract_code(ln)
            continue
        if ln.lower().startswith("total charge"):
            _flush()
            code = None
            continue
        # lines like: "C   -0.35582"
        parts = ln.split()
        if len(parts) >= 2:
            try:
                val = float(parts[-1])
                vals.append(val)
            except Exception:
                pass
    _flush()
    return out

_XPART_RE = re.compile(r"^(?P<name>.+)_X(?P<part>\d+)\.(?P<ext>xyz|XYZ|bo|BO|q|Q)$")
def split_name_part(path: Path):
    m = _XPART_RE.match(path.name)
    if not m:
        return None
    return m.group("name"), int(m.group("part")), m.group("ext").lower()

def find_molecules(raw_dir: Path) -> Dict[str, Dict[str, List[Path]]]:
    """
    Scans raw_dir and groups files by molecule NAME (prefix before '_X').
    Returns dict: name -> {'xyz': [paths...], 'bo': [paths...], 'q': [paths...], 'q_single':[path] }
    Accepts either per-part q or a single 'NAME_X.q'.
    """
    groups: Dict[str, Dict[str, List[Path]]] = {}
    for p in sorted(raw_dir.iterdir()):
        if not p.is_file(): 
            continue
        m = split_name_part(p)
        if m:
            name, part, ext = m
            bucket = groups.setdefault(name, {"xyz": [], "bo": [], "q": [], "q_single": []})
            if ext == "xyz":
                bucket["xyz"].append(p)
            elif ext == "bo":
                bucket["bo"].append(p)
            elif ext == "q":
                # Heuristic: '..._X.q' (no part number in extension area) is parsed as single if filename exactly ends with '_X.q'
                # Our regex already requires '_X<part>.q', so we allow ALSO a separate pattern:
                bucket["q"].append(p)
        else:
            # also allow a global 'NAME_X.q' (whole molecule charges)
            if p.suffix.lower() == ".q" and p.stem.endswith("_X"):
                name = p.stem[:-2]  # strip trailing '_X'
                bucket = groups.setdefault(name, {"xyz": [], "bo": [], "q": [], "q_single": []})
                bucket["q_single"].append(p)
    return groups

# --- main dataset ---

class XYZBOQMultiDataset(InMemoryDataset):
    """
    Layout under {root}/raw:

      NAME_X1.xyz, NAME_X2.xyz, ...      (required: at least one)
      NAME_X1.BO,  NAME_X2.BO,  ...      (optional; read only if read_bo=True)
      NAME_X.q  or NAME_X1.q,...         (optional; read only if read_q=True)
      Optional labels CSV:  NAME_y.csv   (any file '*_y.csv'); must contain a row for 'NAME'
         - first column 'name' or 'id' (string) matching NAME
         - remaining columns are targets -> y (float tensor, shape [T])

    Flags:
      read_q: include charges in node features when present
      read_bo: include edges from .BO when present
      If a requested modality is missing for a molecule, it is simply omitted for that sample.

    Each Data has:
      name (str), pos [N,3], z [N], x [N,F] (F=1 or 2 depending on q),
      (optional) edge_index [2,E], edge_attr [E,1], optional y.
    """
    def __init__(self, root: str, read_q: bool = False, read_bo: bool = False,
                 y_csv: Optional[str] = None, transform=None, pre_transform=None):
        self._read_q = read_q
        self._read_bo = read_bo
        self._y_csv = y_csv
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0],
                                            map_location="cpu",
                                            weights_only=False)

    @property
    def processed_file_names(self):
        return [f"data_q_{self._read_q}_bo_{self._read_bo}.pt"]

    @property
    def raw_file_names(self):
        # Returning an empty list tells PyG there's nothing specific to check
        # before skipping the download step.
        return []
    
    def download(self):
        pass


    def process(self):
        raw_dir = Path(self.raw_dir)

        # 1) Parse all XYZ/BO/q files into unified dicts keyed by CSD_code
        xyz_by_code: Dict[str, Tuple[List[str], np.ndarray]] = {}
        for p in sorted(raw_dir.glob("*.xyz")):
            xyz_by_code.update(read_xyz_multi_by_code(p))
            # also accept uppercase
        for p in sorted(raw_dir.glob("*.XYZ")):
            xyz_by_code.update(read_xyz_multi_by_code(p))

        bo_by_code: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
        if self._read_bo:
            for p in sorted(list(raw_dir.glob("*.BO")) + list(raw_dir.glob("*.bo"))):
                bo_by_code.update(read_bo_multi_by_code(p))

        q_by_code: Dict[str, np.ndarray] = {}
        if self._read_q:
            for p in sorted(list(raw_dir.glob("*.q")) + list(raw_dir.glob("*.Q"))):
                q_by_code.update(read_q_multi_by_code(p))

        if not xyz_by_code:
            raise FileNotFoundError(f"No XYZ blocks with 'CSD_code' found in {raw_dir}")

        # 2) Labels (optional), auto-detect separator (',' or ';')
        name_to_y: Dict[str, np.ndarray] = {}
        if self._y_csv is None:
            cands = list(raw_dir.glob("*.csv"))
            self._y_csv = str(cands[0]) if cands else None
        if self._y_csv:
            # auto-detect delimiter; fall back to semicolon (your file uses ';')
            try:
                df = pd.read_csv(self._y_csv, sep=None, engine="python")
            except Exception:
                df = pd.read_csv(self._y_csv, sep=";")

            # choose ID column
            id_col = "CSD_code" if "CSD_code" in df.columns else df.columns[0]

            # coerce all non-ID columns to numeric, collecting numeric-only targets
            targets_df = df.drop(columns=[id_col]).apply(
                lambda col: pd.to_numeric(col, errors="coerce")
            )
            # drop columns that are completely non-numeric (all NaN)
            targets_df = targets_df.dropna(axis=1, how="all")

            # if you want to keep a subset, you can filter here, e.g.:
            # targets_df = targets_df[["Electronic_E","Dipole_M","HL_Gap"]]  # example

            y_cols = list(targets_df.columns)

            # build mapping
            for _, row in df.iterrows():
                code = str(row[id_col])
                # pull from the numeric view so we never try to cast strings
                y_vals = targets_df.loc[_, y_cols].to_numpy(dtype=np.float32)
                name_to_y[code] = y_vals

        # 3) Build Data objects
        data_list: List[Data] = []
        for code, (symbols, pos) in sorted(xyz_by_code.items()):
            # node features
            z = np.array([sym2Z(s) for s in symbols], dtype=np.int64)
            x_cols = [z.astype(np.float32)[:, None]]
            if self._read_q and code in q_by_code:
                q = q_by_code[code]
                if q.shape[0] == len(symbols):
                    x_cols.append(q.astype(np.float32)[:, None])
            x = np.hstack(x_cols).astype(np.float32)

            # edges
            edge_index = None
            edge_attr = None
            if self._read_bo and code in bo_by_code:
                ei, ea = bo_by_code[code]
                if ei.size:
                    edge_index = torch.from_numpy(ei)
                    edge_attr  = torch.from_numpy(ea)

            # label
            y = None
            if code in name_to_y:
                y = torch.tensor(name_to_y[code], dtype=torch.float32)

            g = Data(
                x=torch.from_numpy(x),
                pos=torch.from_numpy(pos.astype(np.float32)),
                z=torch.from_numpy(z),
                y=y,
                name=code
            )
            if edge_index is not None:
                g.edge_index = edge_index
            if edge_attr is not None:
                g.edge_attr = edge_attr
            data_list.append(g)

        if self.pre_filter is not None:
            data_list = [d for d in data_list if self.pre_filter(d)]
        if self.pre_transform is not None:
            data_list = [self.pre_transform(d) for d in data_list]

        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])



In [37]:
ds = XYZBOQMultiDataset("data/datasets/TMQM_specto/", read_q=False, read_bo=False)
len(ds)

Processing...
Done!


108541

In [38]:
ds[1]

Data(x=[27, 1], y=[8], pos=[27, 3], z=[27], name='ABACAL')

In [35]:
ds[1].y

tensor([-2.2136e+03, -3.3457e-02,  6.1703e+00,  5.7070e-01,  1.4202e-01,
        -1.8905e-01, -4.7030e-02,  1.5807e+02])

In [20]:
ds[0]

Data(x=[288, 1], pos=[288, 3], z=[288], name='tmQM')

In [23]:
len(ds)

1

In [39]:
for idx, mol in enumerate(ds):
    if '_' in mol.name:
        print(idx)
        print(mol)